# Loading Eurostat Data into PostgreSQL

## Project: Germany's Gender Pay Gap (2007–2024)

**Objective:** Connect Python to PostgreSQL, create a raw data table, and load the extracted Eurostat dataset for further transformation using dbt.

## 1. Import Required Libraries

We use SQLAlchemy to connect Python to PostgreSQL and Pandas to transfer our Eurostat DataFrame into the database.

In [35]:
# Install the libraries needed to connect Python to PostgreSQL
%pip install sqlalchemy psycopg2-binary

Note: you may need to restart the kernel to use updated packages.


In [36]:
# Import Pandas for data manipulation
import pandas as pd

# Import SQLAlchemy tools for creating the database connection
from sqlalchemy import create_engine, text

# Import URL encoding to safely handle special characters in passwords
from urllib.parse import quote_plus

In [37]:
# Define the connection details for my local PostgreSQL database
username = "chibuikemaduka"
host = "localhost"
port = "5432"
database = "gender_pay_gap"

# Create the PostgreSQL connection URL
connection_url = (
    f"postgresql+psycopg2://{username}@{host}:{port}/{database}"
)

# Create the SQLAlchemy engine
engine = create_engine(connection_url)

In [38]:
# Test whether Python can successfully communicate with PostgreSQL
with engine.connect() as connection:
    result = connection.execute(text("SELECT version();"))
    print(result.scalar())

PostgreSQL 18.6 (Homebrew) on aarch64-apple-darwin25.6.0, compiled by Apple clang version 21.0.0 (clang-2100.3.34.2), 64-bit


## 2. Create the Raw PostgreSQL Table

I will create a raw table in PostgreSQL to store my original Eurostat observations. This table will serve as the source for subsequent data transformations using dbt.

In [39]:
# Define the SQL query to create my raw Eurostat data table
create_table_query = """
-- Create the raw table for my Eurostat gender pay gap observations
CREATE TABLE IF NOT EXISTS raw_gender_pay_gap (

    -- Store the year of each observation
    year INTEGER NOT NULL,

    -- Store the Eurostat NACE industry identifier
    industry_code VARCHAR(20) NOT NULL,

    -- Store the full industry name
    industry VARCHAR(255) NOT NULL,

    -- Store the reported gender pay gap as a percentage
    gender_pay_gap_percent NUMERIC(6, 2) NOT NULL,

    -- Prevent duplicate records for the same year and industry
    PRIMARY KEY (year, industry_code)
);
"""

# Execute the SQL query using my PostgreSQL connection
with engine.begin() as connection:
    connection.execute(text(create_table_query))

print("Raw data table created successfully.")

Raw data table created successfully.


## 3. Verify the Raw Data Table

I will inspect the newly created PostgreSQL table to confirm that its columns and data types match my intended data structure.

In [40]:
# Retrieve the structure of my newly created PostgreSQL table
with engine.connect() as connection:

    # Query PostgreSQL's information schema for the table columns
    result = connection.execute(text("""
        -- Retrieve column names and data types from my raw table
        SELECT column_name, data_type
        FROM information_schema.columns
        WHERE table_name = 'raw_gender_pay_gap'
        ORDER BY ordinal_position;
    """))

    # Display the table structure
    for row in result:
        print(row)

('frequency', 'character varying')
('unit', 'character varying')
('industry', 'character varying')
('country', 'character varying')
('year', 'integer')
('gender_pay_gap_percent', 'numeric')


## 4. Import the Exported Eurostat Dataset

I will import my exported CSV file into this notebook so that I can transfer the 372 observations into my PostgreSQL database.

In [41]:
# Import my Eurostat dataset from the data folder inside notebooks
df = pd.read_csv("data/eurostat_gender_pay_gap.csv")

# Check the number of observations and columns
print("Dataset shape:", df.shape)

# Display the first five observations
display(df.head())

Dataset shape: (372, 6)


,frequency,unit,industry,country,year,gender_pay_gap_percent
0,Annual,Percentage,Mining and quarrying,Germany,2007,5.1
1,Annual,Percentage,Mining and quarrying,Germany,2008,6.8
2,Annual,Percentage,Mining and quarrying,Germany,2009,10.5
3,Annual,Percentage,Mining and quarrying,Germany,2010,12.3
4,Annual,Percentage,Mining and quarrying,Germany,2011,9.8


In [42]:
# Inspect the data types of all six Eurostat columns
print(df.dtypes)

# Check for missing values in each column
print("\nMissing values:")
print(df.isnull().sum())

frequency                     str
unit                          str
industry                      str
country                       str
year                        int64
gender_pay_gap_percent    float64
dtype: object

Missing values:
frequency                 0
unit                      0
industry                  0
country                   0
year                      0
gender_pay_gap_percent    0
dtype: int64


## 5. Update the Raw PostgreSQL Table

I will update my PostgreSQL table to accommodate all six original Eurostat fields. This ensures that I preserve the source dataset without removing any information.

In [43]:
# Add the frequency column to my existing PostgreSQL table
add_frequency_column = """
-- Add the observation frequency to the raw Eurostat table
ALTER TABLE raw_gender_pay_gap
ADD COLUMN IF NOT EXISTS frequency VARCHAR(20);
"""

# Add the unit column to my existing PostgreSQL table
add_unit_column = """
-- Add the measurement unit to the raw Eurostat table
ALTER TABLE raw_gender_pay_gap
ADD COLUMN IF NOT EXISTS unit VARCHAR(20);
"""

# Execute both SQL statements within one database transaction
with engine.begin() as connection:
    connection.execute(text(add_frequency_column))
    connection.execute(text(add_unit_column))

print("Raw table updated successfully.")

Raw table updated successfully.


In [44]:
# Inspect the constraints currently defined on my PostgreSQL table
with engine.connect() as connection:

    # Retrieve the primary key and other constraints
    result = connection.execute(text("""
        -- Identify the constraints attached to my raw data table
        SELECT constraint_name, constraint_type
        FROM information_schema.table_constraints
        WHERE table_name = 'raw_gender_pay_gap';
    """))

    # Display the constraint information
    for row in result:
        print(row)

('raw_gender_pay_gap_frequency_not_null', 'CHECK')
('raw_gender_pay_gap_unit_not_null', 'CHECK')
('raw_gender_pay_gap_industry_not_null', 'CHECK')
('raw_gender_pay_gap_country_not_null', 'CHECK')
('raw_gender_pay_gap_year_not_null', 'CHECK')
('raw_gender_pay_gap_gender_pay_gap_percent_not_null', 'CHECK')
('raw_gender_pay_gap_pkey', 'PRIMARY KEY')


In [45]:
# Check how many records currently exist in my raw PostgreSQL table
with engine.connect() as connection:

    # Count all records in the raw data table
    result = connection.execute(text("""
        -- Count existing observations before changing the table structure
        SELECT COUNT(*)
        FROM raw_gender_pay_gap;
    """))

    # Display the number of existing records
    print("Existing records:", result.scalar())

Existing records: 372


In [46]:
# Recreate my empty PostgreSQL table with the correct Eurostat structure
recreate_table_query = """
-- Remove the existing empty table and its outdated structure
DROP TABLE IF EXISTS raw_gender_pay_gap;

-- Create a new raw table matching my original Eurostat dataset
CREATE TABLE raw_gender_pay_gap (

    -- Store the observation frequency
    frequency VARCHAR(20) NOT NULL,

    -- Store the measurement unit
    unit VARCHAR(20) NOT NULL,

    -- Store the economic activity or industry
    industry VARCHAR(255) NOT NULL,

    -- Store the country
    country VARCHAR(100) NOT NULL,

    -- Store the observation year
    year INTEGER NOT NULL,

    -- Store the unadjusted gender pay gap percentage
    gender_pay_gap_percent NUMERIC(6, 2) NOT NULL,

    -- Ensure each industry has only one record per year and country
    PRIMARY KEY (country, year, industry)
);
"""

# Execute the SQL statements in a single database transaction
with engine.begin() as connection:
    connection.execute(text(recreate_table_query))

print("Raw PostgreSQL table recreated successfully.")

Raw PostgreSQL table recreated successfully.


## 6. Load Eurostat Data into PostgreSQL

I will transfer my 372 Eurostat observations from Pandas into the raw PostgreSQL table. This will make the original dataset available for SQL queries and subsequent dbt transformations.

In [47]:
# Load my Eurostat DataFrame into the PostgreSQL raw table
df.to_sql(
    name="raw_gender_pay_gap",
    con=engine,
    if_exists="append",
    index=False,
    method="multi"
)

# Confirm that the data transfer is complete
print("Eurostat data successfully loaded into PostgreSQL.")

Eurostat data successfully loaded into PostgreSQL.


## 7. Verify the Data Transfer

I will query PostgreSQL to confirm that all 372 Eurostat observations have been stored successfully in my raw data table.

In [48]:
# Verify the number of observations stored in PostgreSQL
with engine.connect() as connection:

    # Count all records in my raw Eurostat table
    result = connection.execute(text("""
        -- Count the total number of records stored in PostgreSQL
        SELECT COUNT(*)
        FROM raw_gender_pay_gap;
    """))

    # Display the total record count
    total_records = result.scalar()
    print("Records stored in PostgreSQL:", total_records)

Records stored in PostgreSQL: 372


## 8. Validate Data Integrity

I will check my PostgreSQL table for duplicate observations to ensure that each industry has only one record per year and country.

In [49]:
# Check for duplicate country, year and industry combinations
with engine.connect() as connection:

    # Identify combinations that appear more than once
    result = connection.execute(text("""
        -- Group observations and identify duplicate records
        SELECT country, year, industry, COUNT(*) AS record_count
        FROM raw_gender_pay_gap
        GROUP BY country, year, industry
        HAVING COUNT(*) > 1;
    """))

    # Display any duplicate combinations
    duplicates = result.fetchall()

    print("Number of duplicate combinations:", len(duplicates))
    print(duplicates)

Number of duplicate combinations: 0
[]


## 9. Check for Missing Values in PostgreSQL

I will check all six columns in my PostgreSQL table to confirm that no missing values were introduced during the data transfer.

In [50]:
# Check for missing values across all six columns in PostgreSQL
with engine.connect() as connection:

    # Count missing values in each column
    result = connection.execute(text("""
        -- Calculate the number of missing values in every column
        SELECT
            COUNT(*) FILTER (WHERE frequency IS NULL) AS missing_frequency,
            COUNT(*) FILTER (WHERE unit IS NULL) AS missing_unit,
            COUNT(*) FILTER (WHERE industry IS NULL) AS missing_industry,
            COUNT(*) FILTER (WHERE country IS NULL) AS missing_country,
            COUNT(*) FILTER (WHERE year IS NULL) AS missing_year,
            COUNT(*) FILTER (WHERE gender_pay_gap_percent IS NULL)
                AS missing_pay_gap
        FROM raw_gender_pay_gap;
    """))

    # Display the missing-value counts
    print(result.fetchone())

(0, 0, 0, 0, 0, 0)


## 10. Final Validation of My PostgreSQL Data

I will compare the records stored in PostgreSQL with my original Eurostat DataFrame. I will also inspect the year coverage and gender pay gap range to confirm that the data was transferred correctly.

In [51]:
# Retrieve summary statistics directly from PostgreSQL
with engine.connect() as connection:

    # Summarise the number of records, years and pay gap values
    result = connection.execute(text("""
        -- Calculate key statistics for my stored Eurostat dataset
        SELECT
            COUNT(*) AS total_records,
            MIN(year) AS first_year,
            MAX(year) AS last_year,
            MIN(gender_pay_gap_percent) AS minimum_gap,
            MAX(gender_pay_gap_percent) AS maximum_gap,
            ROUND(AVG(gender_pay_gap_percent), 2) AS average_gap
        FROM raw_gender_pay_gap;
    """))

    # Display the database summary
    summary = result.fetchone()
    print("PostgreSQL data summary:", summary)

# Compare the database record count with my original DataFrame
print("\nOriginal DataFrame records:", len(df))
print("PostgreSQL records:", summary[0])
print("Record counts match:", len(df) == summary[0])

PostgreSQL data summary: (372, 2007, 2024, Decimal('-2.20'), Decimal('34.20'), Decimal('17.62'))

Original DataFrame records: 372
PostgreSQL records: 372
Record counts match: True


## 11. PostgreSQL Integration: Conclusion

I successfully connected Python to PostgreSQL and transferred all 372 Eurostat observations into my raw database table.

I validated the data by checking record counts, duplicate combinations, missing values, year coverage and summary statistics. The results matched my original Pandas DataFrame.

My raw dataset is now ready for transformation and testing using dbt.